# NLRexpress on Google Colab

This notebook runs [NLRexpress](https://github.com/eliza-m/NLRexpress) on Google Colab for single-sequence or multi-sequence input. You can provide protein sequences directly, or provide CDS sequences that will be translated with Biopython before running NLRexpress.

The original NLRexpress outputs are left unchanged. MHD D-to-V mutation candidates are written to separate CSV files.

If CDS sequences are provided, the notebook can also write a CodonDomesticate multiple-domestication CSV with `name`, `sequence`, inferred `aa_change`, and MHD candidate metadata columns, so the next notebook can perform domestication and MHD D-to-V mutation in one batch run.


## Suggested Workflow

### If you have CDS sequences

1. Run NLRexpress here with `input_sequence_type = "CDS"`.
2. Download the original NLRexpress result archive.
3. Download the separate MHD D-to-V mutation candidate table.
4. Download the CodonDomesticate multiple-domestication CSV generated from your CDS input.
5. Open the CodonDomesticate notebook and upload that CSV for batch domestication/mutation.

### If you only have protein sequences

1. Run NLRexpress here with `input_sequence_type = "protein"`.
2. Download the original NLRexpress result archive and MHD D-to-V candidate table.
3. CDS domestication cannot be performed until matching CDS sequences are available.

[![Open CodonDomesticate In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YuSugihara/CodonDomesticate/blob/main/notebooks/Domesticate_CDS_Colab.ipynb)


In [ ]:
#@title Get NLRexpress ready
# Install the notebook dependencies and the isolated NLRexpress runtime.
!pip install -q pandas biopython
!apt-get update -qq && apt-get install -y -qq hmmer
!if [ ! -x /content/bin/micromamba ]; then cd /content && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba; fi
!if [ ! -x /content/nlrexpress_env/bin/python ]; then rm -rf /content/nlrexpress_env; /content/bin/micromamba create -y -p /content/nlrexpress_env -c conda-forge python=3.9 click numpy=1.22 pandas=1.4 scipy=1.8 scikit-learn=0.24.2 joblib biopython; fi
!test -x /content/nlrexpress_env/bin/python && echo "Environment ready: /content/nlrexpress_env"
!jackhmmer -h >/dev/null && echo "HMMER ready: $(which jackhmmer)"

# Download NLRexpress and its predictor models if they are not already present.
from pathlib import Path
import os
import subprocess

NLR_REPO = Path("NLRexpress")
MODELS_DIR = NLR_REPO / "models"

if not NLR_REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/eliza-m/NLRexpress.git", str(NLR_REPO)], check=True)
else:
    print("NLRexpress repository already exists.")

if not MODELS_DIR.exists() or not list(MODELS_DIR.glob("*.pkl")):
    subprocess.run(["wget", "-q", "https://nlrexpress.biochim.ro/datasets/models.tar.gz", "-O", str(NLR_REPO / "models.tar.gz")], check=True)
    subprocess.run(["tar", "-xf", str(NLR_REPO / "models.tar.gz"), "-C", str(NLR_REPO)], check=True)
else:
    print("NLRexpress predictor models already exist.")

print("NLRexpress is ready:", NLR_REPO.resolve())

# Load helper functions used by the rest of the notebook.
from pathlib import Path
import csv
import os
import re
import shutil
import subprocess
import textwrap
import zipfile

import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
from google.colab import files

VALID_AA = set("ACDEFGHIKLMNPQRSTVWYBXZJUO*-")
DNA_BASES = set("ACGTUacgtu")
NLR_NOTEBOOK_HELPER_VERSION = "2026-04-28-mhd-probability-threshold"
print("Loaded NLRexpress helper functions:", NLR_NOTEBOOK_HELPER_VERSION)

MICROMAMBA = Path("/content/bin/micromamba")
NLR_ENV_PREFIX = Path("/content/nlrexpress_env")


def clean_protein_sequence(seq):
    seq = "".join(str(seq).split()).upper().replace("-", "")
    invalid = sorted(set(seq) - VALID_AA)
    if invalid:
        raise ValueError(f"Protein sequence contains invalid characters: {''.join(invalid)}")
    if seq.endswith("*"):
        seq = seq[:-1]
    if "*" in seq:
        raise ValueError("Protein sequence contains an internal stop symbol '*'. Remove stop symbols before running NLRexpress.")
    return seq


def clean_cds_sequence(seq):
    cds = "".join(str(seq).split()).upper().replace("U", "T")
    invalid = sorted(set(cds) - set("ACGT"))
    if invalid:
        raise ValueError(f"CDS contains invalid DNA bases: {''.join(invalid)}")
    if len(cds) % 3 != 0:
        raise ValueError("CDS length is not a multiple of 3.")
    return cds


def translate_cds(cds, trim_terminal_stop=True):
    cds = clean_cds_sequence(cds)
    protein = str(Seq(cds).translate(table=1, to_stop=False))
    internal = protein[:-1] if protein.endswith("*") else protein
    if "*" in internal:
        raise ValueError("CDS contains an internal stop codon.")
    if trim_terminal_stop and protein.endswith("*"):
        protein = protein[:-1]
    return protein


def write_fasta(records, fasta_path):
    fasta_path = Path(fasta_path)
    with fasta_path.open("w") as handle:
        for name, seq in records:
            safe_name = re.sub(r"\s+", "_", str(name).strip()) or "sequence"
            clean_seq = clean_protein_sequence(seq)
            handle.write(f">{safe_name}\n")
            for i in range(0, len(clean_seq), 80):
                handle.write(clean_seq[i:i + 80] + "\n")
    return fasta_path


def print_tail(path, title, max_lines=80):
    path = Path(path)
    if not path.exists():
        return
    print(f"\n--- {title}: {path} ---")
    lines = path.read_text(errors="replace").splitlines()
    for line in lines[-max_lines:]:
        print(line)


def run_setup_command(cmd, title):
    print(f"Running setup step: {title}")
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Setup step failed: {title}")


def ensure_hmmer_available():
    if shutil.which("jackhmmer"):
        return
    print("HMMER was not found. Installing it with apt-get.")
    run_setup_command(["apt-get", "update", "-qq"], "apt-get update")
    run_setup_command(["apt-get", "install", "-y", "-qq", "hmmer"], "install hmmer")
    if not shutil.which("jackhmmer"):
        raise RuntimeError("jackhmmer was still not found after installing hmmer.")


def ensure_nlrexpress_environment():
    if not MICROMAMBA.exists():
        raise RuntimeError("micromamba was not found. Run the 'Get NLRexpress ready' cell first.")
    ensure_hmmer_available()
    env_python = NLR_ENV_PREFIX / "bin" / "python"
    if env_python.exists():
        return
    if NLR_ENV_PREFIX.exists():
        print("Removing incomplete NLRexpress environment:", NLR_ENV_PREFIX)
        shutil.rmtree(NLR_ENV_PREFIX)
    print("NLRexpress micromamba environment was not found. Creating it now; this can take several minutes.")
    run_setup_command([
        str(MICROMAMBA), "create", "-y", "-p", str(NLR_ENV_PREFIX),
        "-c", "conda-forge",
        "python=3.9", "click", "numpy=1.22", "pandas=1.4", "scipy=1.8",
        "scikit-learn=0.24.2", "joblib", "biopython",
    ], "create NLRexpress micromamba environment")


def ensure_nlrexpress_files():
    if not Path("NLRexpress/nlrexpress.py").exists():
        raise RuntimeError("NLRexpress source files were not found. Run the 'Get NLRexpress ready' cell first.")
    if not list(Path("NLRexpress/models").glob("*.pkl")):
        raise RuntimeError("NLRexpress predictor models were not found. Run the 'Get NLRexpress ready' cell first.")


def run_nlrexpress(input_fasta, outdir, module="all", outformat="all", cpunum=2):
    ensure_nlrexpress_environment()
    ensure_nlrexpress_files()
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    cmd = [
        str(MICROMAMBA), "run", "-p", str(NLR_ENV_PREFIX),
        "python", "nlrexpress.py",
        "--input", str(Path(input_fasta).resolve()),
        "--outdir", str(outdir.resolve()),
        "--module", module,
        "--outformat", outformat,
        "--cpunum", str(cpunum),
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd="NLRexpress",
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if result.stdout:
        print("\n--- NLRexpress stdout ---")
        print(result.stdout)
    if result.stderr:
        print("\n--- NLRexpress stderr ---")
        print(result.stderr)
    if result.returncode != 0:
        input_stem = Path(input_fasta).stem
        print_tail(outdir / f"{input_stem}.log", "NLRexpress log")
        print_tail(outdir / f"{input_stem}.fasta_proc", "Processed FASTA")
        if module == "all":
            print("\nNLRexpress failed while running module='all'. If you only need MHD D-to-V candidates, try module='nbs' as a diagnostic fallback.")
        raise RuntimeError(f"NLRexpress failed with exit code {result.returncode}. See the stdout/stderr/log above.")
    return outdir


def zip_directory(directory, zip_path):
    directory = Path(directory)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in directory.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(directory.parent))
    return zip_path


def parse_mhd_hits(short_output_path):
    hits = []
    short_output_path = Path(short_output_path)
    if not short_output_path.exists():
        return hits
    with short_output_path.open() as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            if len(parts) < 8:
                continue
            if parts[2] != "MHD":
                continue
            protein_name = parts[0]
            motif_start = int(parts[1])
            probability = float(parts[3])
            motif_seq = parts[7]
            d_offset = motif_seq.rfind("D")
            if d_offset < 0:
                d_offset = len(motif_seq) - 1
            d_position = motif_start + d_offset
            hits.append({
                "name": protein_name,
                "motif_start": motif_start,
                "motif": "MHD",
                "probability": probability,
                "motif_seq": motif_seq,
                "d_position": d_position,
                "aa_change": f"D{d_position}V",
                "source_file": str(short_output_path),
            })
    return hits


def collect_mhd_hits(outdir):
    all_hits = []
    for path in Path(outdir).glob("*.short.output.txt"):
        all_hits.extend(parse_mhd_hits(path))
    return pd.DataFrame(all_hits)


def download_results(outdir, prefix):
    zip_path = zip_directory(outdir, f"{prefix}_nlrexpress_results.zip")
    files.download(str(zip_path))
    return zip_path


def records_from_csv(input_path, sequence_type):
    df = pd.read_csv(input_path)
    if not {"name", "sequence"}.issubset(df.columns):
        raise ValueError("CSV must contain 'name' and 'sequence' columns.")
    protein_records = []
    cds_records = []
    for _, row in df.iterrows():
        name = str(row["name"]).strip()
        sequence = str(row["sequence"])
        if sequence_type == "CDS":
            cds = clean_cds_sequence(sequence)
            protein_records.append((name, translate_cds(cds)))
            cds_records.append({"name": name, "sequence": cds})
        else:
            protein_records.append((name, clean_protein_sequence(sequence)))
    cds_df = pd.DataFrame(cds_records) if cds_records else None
    return protein_records, cds_df


def parse_fasta_records(input_path):
    records = [(record.id, str(record.seq)) for record in SeqIO.parse(str(input_path), "fasta")]
    if not records:
        raise ValueError("No FASTA records found.")
    return records


def records_from_fasta(input_path, sequence_type):
    raw_records = parse_fasta_records(input_path)
    protein_records = []
    cds_records = []
    for name, sequence in raw_records:
        if sequence_type == "CDS":
            cds = clean_cds_sequence(sequence)
            protein_records.append((name, translate_cds(cds)))
            cds_records.append({"name": name, "sequence": cds})
        else:
            protein_records.append((name, clean_protein_sequence(sequence)))
    cds_df = pd.DataFrame(cds_records) if cds_records else None
    return protein_records, cds_df


def format_name_list(names, limit=50):
    names = [str(name) for name in names]
    if not names:
        return "none"
    shown = ", ".join(names[:limit])
    if len(names) > limit:
        shown += f" ... and {len(names) - limit} more"
    return shown


def normalize_mhd_probability_threshold(value):
    if value is None or str(value).strip() == "":
        return None
    threshold = float(value)
    if threshold < 0 or threshold > 100:
        raise ValueError("MHD probability warning threshold must be between 0 and 100, matching the NLRexpress probability column.")
    return threshold


def write_handoff_name_check_report(cds_df, candidate_df, output_csv, min_mhd_probability=None):
    cds_df = cds_df.copy()
    cds_df["name"] = cds_df["name"].astype(str).str.strip()
    threshold = normalize_mhd_probability_threshold(min_mhd_probability)
    if candidate_df is None or candidate_df.empty:
        return None
    candidate_df = candidate_df.copy()
    candidate_df["name"] = candidate_df["name"].astype(str).str.strip()
    if "probability" in candidate_df.columns:
        candidate_df["probability"] = pd.to_numeric(candidate_df["probability"], errors="coerce")
    cds_names = cds_df["name"].astype(str)
    candidate_names = candidate_df["name"].astype(str)
    cds_name_set = set(cds_names)
    candidate_name_set = set(candidate_names)

    cds_without_candidates = sorted(cds_name_set - candidate_name_set)
    candidates_without_cds = sorted(candidate_name_set - cds_name_set)
    duplicate_cds_names = sorted(cds_names[cds_names.duplicated()].unique())
    repeated_candidate_names = sorted(candidate_names[candidate_names.duplicated()].unique())

    print(f"Name check: {len(cds_name_set)} unique CDS name(s), {len(candidate_name_set)} unique MHD-candidate name(s).")
    report_rows = []
    for name in cds_without_candidates:
        report_rows.append({"name": name, "issue": "present_in_cds_missing_from_mhd_candidates"})
    for name in candidates_without_cds:
        report_rows.append({"name": name, "issue": "present_in_mhd_candidates_missing_from_cds"})
    for name in duplicate_cds_names:
        report_rows.append({"name": name, "issue": "duplicate_cds_name"})
    for name in repeated_candidate_names:
        report_rows.append({"name": name, "issue": "multiple_mhd_candidates_highest_probability_will_be_used"})
    low_probability_rows = pd.DataFrame()
    missing_probability_names = []
    if threshold is not None and "probability" in candidate_df.columns:
        low_probability_rows = candidate_df[candidate_df["probability"].lt(threshold)]
        missing_probability_names = sorted(candidate_df.loc[candidate_df["probability"].isna(), "name"].astype(str).unique())
        for _, row in low_probability_rows.iterrows():
            report_rows.append({
                "name": row["name"],
                "issue": "mhd_probability_below_threshold",
                "probability": row.get("probability", ""),
                "probability_threshold": threshold,
                "aa_change": row.get("aa_change", ""),
            })
        for name in missing_probability_names:
            report_rows.append({"name": name, "issue": "mhd_probability_missing_or_non_numeric", "probability_threshold": threshold})

    if cds_without_candidates:
        print(f"WARNING: CDS record(s) without an MHD candidate ({len(cds_without_candidates)}): {format_name_list(cds_without_candidates)}")
    if candidates_without_cds:
        print(f"WARNING: MHD candidate(s) without a matching CDS record ({len(candidates_without_cds)}): {format_name_list(candidates_without_cds)}")
    if duplicate_cds_names:
        print(f"WARNING: Duplicate CDS name(s) were found ({len(duplicate_cds_names)}): {format_name_list(duplicate_cds_names)}")
    if repeated_candidate_names:
        print(f"WARNING: Multiple MHD candidates were found for {len(repeated_candidate_names)} sequence name(s); the highest-probability candidate will be used: {format_name_list(repeated_candidate_names)}")
    if threshold is not None:
        if "probability" not in candidate_df.columns:
            print("WARNING: MHD probability threshold was set, but the candidate table has no probability column.")
        elif not low_probability_rows.empty:
            low_names = sorted(low_probability_rows["name"].astype(str).unique())
            print(f"WARNING: {len(low_probability_rows)} MHD candidate(s) have probability below {threshold}: {format_name_list(low_names)}")
        if missing_probability_names:
            print(f"WARNING: MHD candidate(s) with missing/non-numeric probability: {format_name_list(missing_probability_names)}")

    if not report_rows:
        print("Name check passed: every MHD candidate name matches a CDS record, and every CDS record has an MHD candidate.")
        return None

    report_df = pd.DataFrame(report_rows)
    report_path = Path(output_csv).with_name(Path(output_csv).stem + "_name_check_report.csv")
    report_df.to_csv(report_path, index=False)
    files.download(str(report_path))
    print("Name check report written:", report_path)
    return report_df


def write_codon_domesticate_handoff(cds_df, candidate_df, output_csv, min_mhd_probability=None):
    if cds_df is None or cds_df.empty:
        print("No CDS input was provided, so CodonDomesticate handoff CSV was not created.")
        return None
    threshold = normalize_mhd_probability_threshold(min_mhd_probability)
    cds_df = cds_df.copy()
    cds_df["name"] = cds_df["name"].astype(str).str.strip()
    if candidate_df is None or candidate_df.empty:
        print("WARNING: No MHD motif was found in the NLRexpress result. The CodonDomesticate batch input CSV will be written with an empty aa_change column.")
        batch_table = cds_df.copy()
        batch_table["aa_change"] = ""
        if threshold is not None:
            batch_table["mhd_probability_warning_threshold"] = threshold
    else:
        candidate_df = candidate_df.copy()
        candidate_df["name"] = candidate_df["name"].astype(str).str.strip()
        if "probability" in candidate_df.columns:
            candidate_df["probability"] = pd.to_numeric(candidate_df["probability"], errors="coerce")
        write_handoff_name_check_report(cds_df, candidate_df, output_csv, min_mhd_probability=threshold)
        first_candidates = candidate_df.sort_values(["name", "probability"], ascending=[True, False]).drop_duplicates("name")
        first_candidates = first_candidates.rename(columns={"aa_change": "mhd_dv_aa_change"})
        candidate_columns = [col for col in first_candidates.columns if col != "name"]
        batch_table = cds_df.merge(first_candidates[["name"] + candidate_columns], on="name", how="left")
        batch_table["aa_change"] = batch_table["mhd_dv_aa_change"].fillna("")
        batch_table = batch_table.drop(columns=["mhd_dv_aa_change"])
        if threshold is not None:
            batch_table["mhd_probability_warning_threshold"] = threshold
            if "probability" in batch_table.columns:
                selected_probability = pd.to_numeric(batch_table["probability"], errors="coerce")
                batch_table["mhd_probability_below_threshold"] = selected_probability.lt(threshold).fillna(False)
                low_selected_names = batch_table.loc[batch_table["mhd_probability_below_threshold"], "name"].astype(str).tolist()
                if low_selected_names:
                    print(f"WARNING: Selected MHD candidate(s) below probability threshold {threshold}; review before CodonDomesticate: {format_name_list(low_selected_names)}")
        missing_names = batch_table.loc[batch_table["aa_change"].eq(""), "name"].astype(str).tolist()
        if missing_names:
            print(f"WARNING: No MHD motif was found for {len(missing_names)} CDS record(s): {format_name_list(missing_names)}. Their aa_change values will be empty.")
    batch_table.to_csv(output_csv, index=False)
    files.download(output_csv)
    print("Ready for CodonDomesticate batch input:", output_csv)
    return batch_table


## Single Sequence Run

Paste one protein or CDS sequence and run NLRexpress. Use `module = "nbs"` if you only need NB-ARC/NBS motifs including MHD.


In [ ]:
#@title Configure single sequence input
input_sequence_type = "CDS" #@param ["CDS", "protein"]
sequence_name = "example_NLR" #@param {type:"string"}
sequence = "" #@param {type:"string"}
module = "all" #@param ["all", "nbs", "cc", "tir", "lrr"]
outformat = "all" #@param ["all", "short", "long"]
cpunum = 2 #@param {type:"integer"}


In [ ]:
#@title Run NLRexpress for one sequence
if not sequence.strip():
    raise ValueError("Please paste a protein or CDS sequence into sequence.")
sequence = "".join(str(sequence).split())

single_cds_df = None
if input_sequence_type == "CDS":
    single_cds = clean_cds_sequence(sequence)
    single_protein = translate_cds(single_cds)
    single_cds_df = pd.DataFrame([{"name": sequence_name, "sequence": single_cds}])
else:
    single_protein = clean_protein_sequence(sequence)

single_fasta = write_fasta([(sequence_name, single_protein)], "single_input.fa")
single_outdir = run_nlrexpress(single_fasta, "nlrexpress_single_output", module=module, outformat=outformat, cpunum=cpunum)

single_mhd_df = collect_mhd_hits(single_outdir)
print("MHD D-to-V candidates:")
display(single_mhd_df if not single_mhd_df.empty else pd.DataFrame(columns=["name", "motif_start", "motif_seq", "d_position", "aa_change", "probability"]))

single_mhd_path = "single_mhd_dv_candidates.csv"
single_mhd_df.to_csv(single_mhd_path, index=False)
files.download(single_mhd_path)

if single_cds_df is not None:
    single_handoff_df = write_codon_domesticate_handoff(single_cds_df, single_mhd_df, "single_codon_domesticate_input_with_mhd_dv.csv")
    display(single_handoff_df)

download_results(single_outdir, "single")


## Multiple Sequence Run

Upload a CSV with `name` and `sequence` columns, or upload a FASTA file containing one or more sequences. Set `multi_input_sequence_type` to `CDS` if the uploaded sequences are CDS; the notebook will translate them with Biopython before running NLRexpress and keep the CDS for CodonDomesticate handoff.


In [ ]:
#@title Download multi-sequence CSV templates
protein_template_path = "nlrexpress_protein_template.csv"
with open(protein_template_path, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "sequence"])
    writer.writeheader()
    writer.writerow({"name": "example_NLR_1", "sequence": "MA..."})
files.download(protein_template_path)

cds_template_path = "nlrexpress_cds_template.csv"
with open(cds_template_path, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "sequence"])
    writer.writeheader()
    writer.writerow({"name": "example_NLR_1", "sequence": "ATG..."})
files.download(cds_template_path)

print("Templates written:", protein_template_path, cds_template_path)


In [ ]:
#@title Upload sequence CSV or FASTA
uploaded = files.upload()
input_path = next(iter(uploaded))
print("Uploaded:", input_path)


In [ ]:
#@title Configure multiple sequence run
multi_input_sequence_type = "CDS" #@param ["CDS", "protein"]
multi_module = "all" #@param ["all", "nbs", "cc", "tir", "lrr"]
multi_outformat = "all" #@param ["all", "short", "long"]
multi_cpunum = 2 #@param {type:"integer"}


In [ ]:
#@title Run NLRexpress for multiple sequences
input_path = Path(input_path)
if input_path.suffix.lower() == ".csv":
    protein_records, multi_cds_df = records_from_csv(input_path, multi_input_sequence_type)
else:
    protein_records, multi_cds_df = records_from_fasta(input_path, multi_input_sequence_type)

multi_fasta = write_fasta(protein_records, "multi_input.fa")
multi_outdir = run_nlrexpress(multi_fasta, "nlrexpress_multi_output", module=multi_module, outformat=multi_outformat, cpunum=multi_cpunum)

multi_mhd_df = collect_mhd_hits(multi_outdir)
print("MHD D-to-V candidates:")
display(multi_mhd_df if not multi_mhd_df.empty else pd.DataFrame(columns=["name", "motif_start", "motif_seq", "d_position", "aa_change", "probability"]))

multi_mhd_path = "multi_mhd_dv_candidates.csv"
multi_mhd_df.to_csv(multi_mhd_path, index=False)
files.download(multi_mhd_path)

if multi_cds_df is not None:
    multi_handoff_df = write_codon_domesticate_handoff(multi_cds_df, multi_mhd_df, "multi_codon_domesticate_input_with_mhd_dv.csv")
    display(multi_handoff_df)

download_results(multi_outdir, "multi")


## Prepare CodonDomesticate Input with MHD D-to-V Mutations

If you ran NLRexpress with CDS input, the notebook already creates a CodonDomesticate multiple-domestication CSV. Use this optional section when you want to regenerate that CSV from data already in the Colab session, or when you ran NLRexpress with protein input and need to upload a separate matching CDS CSV afterward. The `name` values should match the protein names in the NLRexpress MHD table. Extra NLRexpress columns such as motif position and probability are kept in the CSV; CodonDomesticate batch mode uses `name`, `sequence`, and `aa_change` and ignores the extra columns.

The notebook prints warnings when no MHD motif is detected, when a CDS has no matching MHD candidate, when an MHD candidate has no matching CDS, or when multiple MHD candidates are found for one sequence. For multiple candidates, the highest-probability candidate is used and a name-check report CSV is written.


In [ ]:
#@title Optional: prepare CodonDomesticate batch table from current session
manual_mhd_probability_warning_threshold = 90 #@param {type:"number"}

candidate_df = None
if "multi_mhd_df" in globals() and multi_mhd_df is not None:
    candidate_df = multi_mhd_df
    candidate_source = "multi_mhd_df"
elif "single_mhd_df" in globals() and single_mhd_df is not None:
    candidate_df = single_mhd_df
    candidate_source = "single_mhd_df"
else:
    raise ValueError("No MHD candidate table is available. Run NLRexpress first.")

cds_df = None
if "multi_cds_df" in globals() and multi_cds_df is not None and not multi_cds_df.empty:
    cds_df = multi_cds_df.copy()
    cds_source = "multi_cds_df"
elif "single_cds_df" in globals() and single_cds_df is not None and not single_cds_df.empty:
    cds_df = single_cds_df.copy()
    cds_source = "single_cds_df"
else:
    print("No CDS table was found in this Colab session. Upload a matching CDS CSV with name and sequence columns.")
    uploaded_cds = files.upload()
    cds_csv_path = next(iter(uploaded_cds))
    cds_df = pd.read_csv(cds_csv_path)
    cds_source = cds_csv_path

if not {"name", "sequence"}.issubset(cds_df.columns):
    raise ValueError("CDS CSV must contain 'name' and 'sequence' columns.")
cds_df = cds_df.copy()
cds_df["name"] = cds_df["name"].astype(str).str.strip()
cds_df["sequence"] = cds_df["sequence"].map(clean_cds_sequence)
print(f"Using CDS from {cds_source} and MHD candidates from {candidate_source}.")

manual_handoff_df = write_codon_domesticate_handoff(cds_df, candidate_df, "codon_domesticate_input_with_mhd_dv.csv", min_mhd_probability=manual_mhd_probability_warning_threshold)
display(manual_handoff_df)
